In [2]:

import numpy as np
import pandas as pd


In [38]:

def calculate_strata_params(df):

    # Prevent dataframe mutation
    df = df.copy()

    # Variance of the data sample
    sample_var = df['y'].var()

    # Variance for each strata in the sample
    strata_vars = df.groupby('strata')['y'].var()

    # Size of each strata
    strata_sizes = df['strata'].value_counts()

    # Percentage of each strata in population
    strata_weights = df['strata'].value_counts(normalize=True)

    # Min strata percentage
    min_weight = strata_weights.min()

    # Stratifed variance
    stratified_var = (strata_vars * strata_weights).sum()

    # Compilate dictionary of parameters
    params = {
        'sample_var' : sample_var,
        'strata_vars' : strata_vars,
        'strata_sizes' : strata_sizes,
        'strata_weights' : strata_weights,
        'min_weight' : min_weight,
        'stratified_var' : stratified_var
    }

    return params


def print_strata_params(df):
    params = calculate_strata_params(df)
    print(f'sample_var={params['sample_var']}; stratified_var={params['stratified_var']:0.0f}; min_weight={params['min_weight']*100:0.2f}%')


def get_feature_with_min_stratified_variance(df):

    # Prevent dataframe mutation
    df = df.copy()

    # Generate feature names
    features = [f'x{x}' for x in range(1, 11)]

    rows = []
    for feature in features:
        # Print feature name and add up to 5 spaces
        # print(f'{feature:<5}', end='')

        # Stratify sample data by feature values
        df['strata'] = df[feature]

        # Print results
        params = calculate_strata_params(df)
        row = {
            'feature' : feature,
            'sample_var' : params['sample_var'],
            'stratified_var' : params['stratified_var'],
            'min_weight' : params['min_weight']
        }
        rows.append(row)

    features_df = pd.DataFrame(rows)
    features_df = features_df.sort_values('stratified_var')
    print('Features with minimal stratified variance:')
    print(features_df)
    print()

    return features_df.iloc[0, 0]


def get_feature_values_to_unite(df, feature, metric='y'):
    # Group by feature and aggregates
    df_agg = df.groupby(feature)[metric].agg(['count', 'mean'])

    # Calculate percent of each strata in data sample
    df_agg['percent'] = (df_agg['count'] / len(df)).round(3)

    # Round mean value
    df_agg['mean'] = df_agg['mean'].round(1)

    # Sort to get smallest percents
    df_agg = df_agg.sort_values('percent')

    # Get cumulative sum
    df_agg['percent_cumsum'] = df_agg['percent'].cumsum()
    df_agg['unite'] = (df_agg['percent'] < 0.05) | (df_agg['percent'] < 0.05)
    print('\nStratas percentage:')
    print(df_agg)

    # Filter
    df_agg = df_agg[df_agg['unite']]
    df_agg = df_agg.reset_index()
    print('\nStratas filtered:')
    print(df_agg)

    return df_agg[feature]


def unite_feature_values(df, feature, to_unite):
    df['strata'] = [to_unite.min() if v in to_unite.values else v for v in df[feature]]
    return df

"""
Есть набор признаков, которые вычисляются независимо от эксперимента. ?
Используя эти признаки, нужно разбить объекты на страты так,
чтобы дисперсия стратифицированного среднего была минимальна и доля каждой страты была не менее 5% от всех данныx.

Данные разбиты на 2 части.
Решение будет проверяться на второй части данных.
Значения в столбцах x1, ..., x10 — признаки, которые можно использовать для вычисления страт.
Значения в столбце y — измерения, по которым будет вычисляться целевая метрика эксперимента.

Дисперсия должна не превышать 50000.
"""

def get_strats(df_features):
    """Возвращает страты объектов.

    :param df_features (pd.DataFrame): таблица с признаками x1, ..., x10
    :return (list | np.array | pd.Series): список страт объектов размера len(df).
    """
    # YOUR_CODE_HERE
    pass


df = pd.read_csv('data/chapter_08_stratification_task_data_public.csv')

feature_min_var = get_feature_with_min_stratified_variance(df)

feature_min_var = 'x2'

feature_values_to_unite = get_features_to_unite(df, feature_min_var, 'y')

df_united = unite_feature_values(df, feature_min_var, feature_values_to_unite)

params = calculate_strata_params(df_united)

print(params)


Features with minimal stratified variance:
  feature    sample_var  stratified_var  min_weight
1      x2  66078.094333    47232.075241      0.0001
6      x7  66078.094333    48054.669337      0.0001
9     x10  66078.094333    63755.711464      0.0007
2      x3  66078.094333    64390.510806      0.0001
7      x8  66078.094333    64858.815656      0.3495
5      x6  66078.094333    65713.855316      0.0001
4      x5  66078.094333    66021.751410      0.4074
8      x9  66078.094333    66050.549744      0.3970
3      x4  66078.094333    66054.371201      0.3261
0      x1  66078.094333    66233.928543      0.0002


Stratas percentage:
    count    mean  percent  percent_cumsum  unite
x2                                               
12      2   572.5    0.000           0.000   True
47      2   518.0    0.000           0.000   True
46      3   851.7    0.000           0.000   True
45      4   740.5    0.000           0.000   True
48      1   966.0    0.000           0.000   True
49      1   6